In [ ]:
# Cell 1 -- install deps, auth to Hugging Face via Colab Secrets, download coding-v1 dataset.
# HF_TOKEN must be added as a Colab Secret (key icon in the left sidebar -> "Add new
# secret", name it HF_TOKEN, paste your HF write token, enable "Notebook access") before
# this cell runs -- MANUAL_ACTION_REQUIRED, Colab has no API to set secrets, and no API
# to run this notebook unattended either -- you drive execution here, unlike Kaggle's
# push-based batch commits.
!pip install -q -U transformers peft bitsandbytes accelerate trl huggingface_hub

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

from huggingface_hub import login, hf_hub_download
login(token=hf_token)

train_path = hf_hub_download(
    repo_id="makremlupin/ai-software-engineer-artifacts", repo_type="dataset",
    filename="datasets/coding-v1/train.jsonl", token=hf_token,
)
eval_path = hf_hub_download(
    repo_id="makremlupin/ai-software-engineer-artifacts", repo_type="dataset",
    filename="datasets/coding-v1/eval.jsonl", token=hf_token,
)
print("TRAIN_PATH:", train_path)
print("EVAL_PATH:", eval_path)


In [ ]:
# Cell 2 -- load base model in 4-bit (QLoRA) and attach a LoRA adapter.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Cell 3 -- build tokenized datasets with the loss masked to assistant tokens only.
# Qwen's chat template does not support the native {% generation %} mask (verified
# locally before this was written -- return_assistant_tokens_mask came back all zeros),
# so the prompt-only prefix is tokenized separately and its length is used to mask labels.
import json
from datasets import Dataset

MAX_LENGTH = 2048  # covers the longest real example (1568 tokens), verified locally

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def tokenize_example(example):
    messages = example["messages"]
    full = tokenizer.apply_chat_template(messages, tokenize=True)
    full_ids = full["input_ids"]
    prefix = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
    prefix_len = len(prefix["input_ids"])

    full_ids = full_ids[:MAX_LENGTH]
    labels = [-100] * min(prefix_len, len(full_ids)) + full_ids[prefix_len:MAX_LENGTH]
    labels = labels[:len(full_ids)]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

train_raw = load_jsonl(train_path)
eval_raw = load_jsonl(eval_path)

train_ds = Dataset.from_list(train_raw).map(tokenize_example, remove_columns=["messages", "task_id", "category", "source_model"])
eval_ds = Dataset.from_list(eval_raw).map(tokenize_example, remove_columns=["messages", "task_id", "category", "source_model"])

print(f"train examples: {len(train_ds)}, eval examples: {len(eval_ds)}")
print("sample labels (first 20, -100 = masked):", train_ds[0]["labels"][:20])


In [ ]:
# Cell 4 -- train with QLoRA, uploading a checkpoint to HF storage after every save so a
# killed session can resume from the last uploaded checkpoint. On Colab this is not just a
# deliberate test -- free-tier sessions can genuinely disconnect on idle/timeout, so this
# resilience is load-bearing, not hypothetical.
import os
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, TrainerCallback
from huggingface_hub import HfApi, list_repo_files

HF_REPO_ID = "makremlupin/ai-software-engineer-artifacts"
CHECKPOINT_PREFIX = "checkpoints/coding-v1-qwen7b-qlora"
OUTPUT_DIR = "/content/checkpoints"

api = HfApi()

class UploadCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = f"{OUTPUT_DIR}/checkpoint-{state.global_step}"
        if os.path.isdir(ckpt_dir):
            api.upload_folder(
                folder_path=ckpt_dir,
                path_in_repo=f"{CHECKPOINT_PREFIX}/checkpoint-{state.global_step}",
                repo_id=HF_REPO_ID, repo_type="dataset", token=hf_token,
            )
            api.upload_file(
                path_or_fileobj=str(state.global_step).encode(),
                path_in_repo=f"{CHECKPOINT_PREFIX}/latest_step.txt",
                repo_id=HF_REPO_ID, repo_type="dataset", token=hf_token,
            )
            print(f"UPLOADED_CHECKPOINT step={state.global_step}")

resume_from = None
try:
    remote_files = list_repo_files(HF_REPO_ID, repo_type="dataset", token=hf_token)
    if f"{CHECKPOINT_PREFIX}/latest_step.txt" in remote_files:
        latest_step_path = hf_hub_download(
            repo_id=HF_REPO_ID, repo_type="dataset",
            filename=f"{CHECKPOINT_PREFIX}/latest_step.txt", token=hf_token,
        )
        latest_step = open(latest_step_path).read().strip()
        local_ckpt_dir = f"{OUTPUT_DIR}/checkpoint-{latest_step}"
        ckpt_files = [f for f in remote_files if f.startswith(f"{CHECKPOINT_PREFIX}/checkpoint-{latest_step}/")]
        os.makedirs(local_ckpt_dir, exist_ok=True)
        for remote_f in ckpt_files:
            local_f = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=remote_f, token=hf_token)
            fname = os.path.basename(remote_f)
            os.link(local_f, f"{local_ckpt_dir}/{fname}") if not os.path.exists(f"{local_ckpt_dir}/{fname}") else None
        resume_from = local_ckpt_dir
        print(f"RESUMING_FROM_CHECKPOINT step={latest_step}")
except Exception as e:
    print(f"NO_CHECKPOINT_TO_RESUME: {e}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=1,
    save_strategy="steps",
    save_steps=4,
    eval_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
    callbacks=[UploadCheckpointCallback()],
)

trainer.train(resume_from_checkpoint=resume_from)


In [ ]:
# Cell 5 -- final eval, upload the trained adapter, print a DONE marker for the local
# machine to poll for (Colab cannot reach our local Postgres registry directly).
eval_metrics = trainer.evaluate()
print("FINAL_EVAL_METRICS:", eval_metrics)

ADAPTER_PREFIX = "adapters/coding-v1-qwen7b-qlora"
model.save_pretrained("/content/final_adapter")
tokenizer.save_pretrained("/content/final_adapter")

api.upload_folder(
    folder_path="/content/final_adapter",
    path_in_repo=ADAPTER_PREFIX,
    repo_id="makremlupin/ai-software-engineer-artifacts", repo_type="dataset", token=hf_token,
)
print("TRAINING_DONE")
print("ADAPTER_LOCATION:", "makremlupin/ai-software-engineer-artifacts/" + ADAPTER_PREFIX)


In [ ]:
# Cell 6 -- base-vs-fine-tuned evaluation on the SAME held-out eval set, same loss metric.
# No separate inference-serving session needed: peft's disable_adapter() context toggles
# LoRA off on the already-loaded model, so both numbers come from the identical weights
# in memory, just with/without the adapter -- directly comparable, real scores.
import math

finetuned_eval = eval_metrics  # already computed in the previous cell, adapter enabled

with model.disable_adapter():
    base_eval = trainer.evaluate()

def perplexity(loss):
    return math.exp(loss) if loss is not None and loss < 20 else float("inf")

results = {
    "base_eval_loss": base_eval.get("eval_loss"),
    "base_perplexity": perplexity(base_eval.get("eval_loss")),
    "finetuned_eval_loss": finetuned_eval.get("eval_loss"),
    "finetuned_perplexity": perplexity(finetuned_eval.get("eval_loss")),
    "eval_examples": len(eval_ds),
    "base_model": "Qwen/Qwen2.5-Coder-7B-Instruct",
}

print("BASE_EVAL:", base_eval)
print("FINETUNED_EVAL:", finetuned_eval)
print("EVAL_RESULTS:", results)

import json as _json
with open("/content/evaluation_results.json", "w") as f:
    _json.dump(results, f)

api.upload_file(
    path_or_fileobj="/content/evaluation_results.json",
    path_in_repo="evaluations/coding-v1-qwen7b-qlora/results.json",
    repo_id="makremlupin/ai-software-engineer-artifacts", repo_type="dataset", token=hf_token,
)
print("EVALUATION_DONE")
